# Launcher bootstrap offline v5

Ce notebook sert a lancer et inspecter le pipeline bootstrap v5 depuis Jupyter. Les sorties restent offline: rien n'est branche au bot live, et les labels ne sont pas des labels GTO.


## Garde-fous

- Pipeline cible: v5 dist-aligned uniquement.
- Les warnings `not_gto`, `not_for_production`, `contains_weak_rule_labels` et `call_class_absent` restent des signaux a surveiller.
- Des metriques parfaites sont interpretees comme signal d'overfit probable, pas comme preuve de qualite strategique.
- Le run `v5_4000` ecrit dans des dossiers separes et ne remplace pas le baseline `v5`.


In [69]:
from pathlib import Path
import csv
import json
import subprocess
import sys
from collections import Counter, defaultdict

ROOT = Path.cwd()
PYTHON = sys.executable

V5_CANDIDATES = ROOT / "outputs/readiness/bootstrap_candidate_dataset_v5/candidates.csv"
V5_MODEL_DIR = ROOT / "outputs/readiness/bootstrap_model_v5"
V5_TRAINING_REPORT = V5_MODEL_DIR / "training_report_v5.json"
V5_4000_DIR = ROOT / "outputs/readiness/bootstrap_candidate_dataset_v5_4000"
V5_4000_CANDIDATES = V5_4000_DIR / "candidates.csv"
V5_4000_MODEL_DIR = ROOT / "outputs/readiness/bootstrap_model_v5_4000"
V5_4000_TRAINING_REPORT = V5_4000_MODEL_DIR / "training_report.json"
V5_4000_COMPARISON = V5_4000_MODEL_DIR / "comparison_v5_vs_v5_4000.json"

def run_cmd(args, *, timeout=None):
    print("$", " ".join(str(part) for part in args))
    completed = subprocess.run(
        [str(part) for part in args],
        cwd=ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"Commande echouee: {completed.returncode}")
    return completed.stdout

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def load_csv(path):
    with Path(path).open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

def is_true(value):
    return str(value).strip().lower() in {"true", "1", "yes"}

print("Projet:", ROOT)
print("Python:", PYTHON)


Projet: c:\Users\Éleve\Pictures\Traine_aide_decission
Python: c:\Users\Éleve\AppData\Local\Programs\Python\Python313\python.exe


## Lancements

Execute seulement les cellules dont tu as besoin. Le run `v5_4000` peut etre long; le training v5 simple est plus court.


## Training v5 dist-aligned

Cette cellule entraine le modele offline sur `outputs/readiness/bootstrap_candidate_dataset_v5/candidates.csv`. Elle ne branche rien au bot live et n'ajoute pas `CALL`.


In [70]:
train = True  # Mettre True pour lancer toute la suite pytest depuis le notebook.
RUN_V5_TRAINING = train

if RUN_V5_TRAINING:
    run_cmd([
        PYTHON,
        "experiments/train_bootstrap_v5.py",
        "--input", V5_CANDIDATES,
        "--output-dir", V5_MODEL_DIR,
        "--min-rows", 100,
        "--random-seed", 17,
    ], timeout=120)
else:
    print("Training v5 ignore. Passe RUN_V5_TRAINING=True pour le lancer.")


$ c:\Users\Éleve\AppData\Local\Programs\Python\Python313\python.exe experiments/train_bootstrap_v5.py --input c:\Users\Éleve\Pictures\Traine_aide_decission\outputs\readiness\bootstrap_candidate_dataset_v5\candidates.csv --output-dir c:\Users\Éleve\Pictures\Traine_aide_decission\outputs\readiness\bootstrap_model_v5 --min-rows 100 --random-seed 17
c:\Users\Ã‰leve\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
{
  "accuracy": 1.0,
  "bot_liv

In [71]:
if V5_TRAINING_REPORT.exists():
    report = load_json(V5_TRAINING_REPORT)
    print("Rapport:", V5_TRAINING_REPORT)
    print("Dataset:", report["dataset_size"])
    print("Features modele:")
    for feature in report["model_feature_columns"]:
        print("-", feature)
    print("Leakage utilise:", report["leakage_columns_used_by_model"])
    print("Ignored columns:", len(report["ignored_columns"]))
else:
    print("Rapport v5 absent:", V5_TRAINING_REPORT)


Rapport: c:\Users\Éleve\Pictures\Traine_aide_decission\outputs\readiness\bootstrap_model_v5\training_report_v5.json
Dataset: {'excluded': 246, 'total': 747, 'usable': 501}
Features modele:
- features.pot
- features.to_call
- features.to_call_pot_ratio
- features.equity_table
- features.equity_1v1
- features.equity_known
- features.equity_required
- features.equity_gap
- features.ev
- features.call_max
- features.board_card_count
- features.hero_cards_known
- features.hero_stack
- features.effective_stack
- features.stack_to_pot_ratio
- features.has_check
- features.has_call
- features.hero_position
Leakage utilise: []
Ignored columns: 72


## Generation large v5_4000

Cette cellule cree un dataset et un modele separes dans `bootstrap_candidate_dataset_v5_4000` et `bootstrap_model_v5_4000`. Le baseline `bootstrap_model_v5` n'est pas ecrase.


In [72]:
RUN_V5_4000 = train
nb_solve_target = 1000 # 4000
if RUN_V5_4000:
    run_cmd([
        PYTHON,
        "experiments/run_bootstrap_v5_4000.py",
        "--output-dir", V5_4000_DIR,
        "--model-dir", V5_4000_MODEL_DIR,
        "--baseline-model-dir", V5_MODEL_DIR,
        "--target-solves", nb_solve_target,
        "--min-usable-rows", nb_solve_target,
        "--class-floor", nb_solve_target // 10,
        "--min-training-rows", nb_solve_target // 40,
        "--random-seed", 17,
    ], timeout=1800)
else:
    print("Generation v5_4000 ignoree. Passe RUN_V5_4000=True pour la lancer.")


$ c:\Users\Éleve\AppData\Local\Programs\Python\Python313\python.exe experiments/run_bootstrap_v5_4000.py --output-dir c:\Users\Éleve\Pictures\Traine_aide_decission\outputs\readiness\bootstrap_candidate_dataset_v5_4000 --model-dir c:\Users\Éleve\Pictures\Traine_aide_decission\outputs\readiness\bootstrap_model_v5_4000 --baseline-model-dir c:\Users\Éleve\Pictures\Traine_aide_decission\outputs\readiness\bootstrap_model_v5 --target-solves 1000 --min-usable-rows 1000 --class-floor 100 --min-training-rows 25 --random-seed 17
c:\Users\Ã‰leve\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for al

In [73]:
if V5_4000_COMPARISON.exists():
    comparison = load_json(V5_4000_COMPARISON)
    report = load_json(V5_4000_DIR / "dataset_report.json")
    training = load_json(V5_4000_TRAINING_REPORT)
    print("Dataset:", V5_4000_CANDIDATES)
    print("Modele:", V5_4000_MODEL_DIR / "model.joblib")
    print("dataset size:", training["dataset_size"])
    print("classes:", training["label_distribution"])
    print("leakage used:", training["leakage_columns_used_by_model"])
    print("dist sample prediction:", comparison["dist_sample_prediction"]["prediction_status"], comparison["dist_sample_prediction"].get("prediction"))
    print("baseline non ecrase:", comparison["baseline_v5_not_overwritten"])
else:
    print("Comparaison v5_4000 absente:", V5_4000_COMPARISON)


Dataset: c:\Users\Éleve\Pictures\Traine_aide_decission\outputs\readiness\bootstrap_candidate_dataset_v5_4000\candidates.csv
Modele: c:\Users\Éleve\Pictures\Traine_aide_decission\outputs\readiness\bootstrap_model_v5_4000\model.joblib
dataset size: {'excluded': 510, 'total': 1512, 'usable': 1002}
classes: {'CHECK': 334, 'FOLD': 334, 'RAISE': 334}
leakage used: []
dist sample prediction: ok FOLD
baseline non ecrase: True


In [74]:
RUN_TESTS = train  # Mettre True pour lancer toute la suite pytest depuis le notebook.

if RUN_TESTS:
    run_cmd([PYTHON, "-m", "pytest"], timeout=180)
else:
    print("Tests ignores. Passe RUN_TESTS=True pour verifier le repo.")


$ c:\Users\Éleve\AppData\Local\Programs\Python\Python313\python.exe -m pytest
============================= test session starts =============================
platform win32 -- Python 3.13.2, pytest-8.3.4, pluggy-1.6.0
rootdir: c:\Users\Ã‰leve\Pictures\Traine_aide_decission
configfile: pytest.ini
testpaths: tests
plugins: anyio-4.13.0
collected 251 items

tests\test_action_candidate.py ..........                                [  3%]
tests\test_bootstrap_prediction.py .....                                 [  5%]
tests\test_bootstrap_training.py ...........                             [ 10%]
tests\test_bootstrap_v4_audit.py .....                                   [ 12%]
tests\test_bootstrap_v4_generation.py ...                                [ 13%]
tests\test_bootstrap_v5_4000.py .                                        [ 13%]
tests\test_bootstrap_v5_advanced_diagnostics.py ..                       [ 14%]
tests\test_bootstrap_v5_training.py ..                                   [ 15%]
tes

## Inspection rapide

Cette section lit les artefacts v5 deja produits et affiche les points utiles pour juger l'overfit et la qualite du dataset.


In [75]:
summary_files = [
    ("Dataset v5_4000", V5_4000_DIR / "dataset_report.json"),
    ("Modele v5_4000", V5_4000_TRAINING_REPORT),
    ("Comparaison v5", V5_4000_COMPARISON),
    ("Modele v5", V5_TRAINING_REPORT),
]

for title, path in summary_files:
    print("\n==", title, "==")
    if not path.exists():
        print("Absent:", path)
        continue
    payload = load_json(path)
    print("Fichier:", path)
    for key in [
        "status",
        "schema",
        "target_solves",
        "total_spots_generated",
        "usable_rows",
        "rejected_spots",
        "root_player_not_hero_errors",
        "critical_warnings",
        "dataset_size",
        "label_distribution",
        "leakage_columns_used_by_model",
        "baseline_v5_not_overwritten",
    ]:
        if key in payload:
            print(f"{key}:", payload[key])



== Dataset v5_4000 ==
Fichier: c:\Users\Éleve\Pictures\Traine_aide_decission\outputs\readiness\bootstrap_candidate_dataset_v5_4000\dataset_report.json
status: ok
schema: dist_aligned_v5
target_solves: 1000
total_spots_generated: 1000
usable_rows: 1002
rejected_spots: 510
root_player_not_hero_errors: 0
critical_warnings: []

== Modele v5_4000 ==
Fichier: c:\Users\Éleve\Pictures\Traine_aide_decission\outputs\readiness\bootstrap_model_v5_4000\training_report.json
status: ok
dataset_size: {'excluded': 510, 'total': 1512, 'usable': 1002}
label_distribution: {'CHECK': 334, 'FOLD': 334, 'RAISE': 334}
leakage_columns_used_by_model: []

== Comparaison v5 ==
Fichier: c:\Users\Éleve\Pictures\Traine_aide_decission\outputs\readiness\bootstrap_model_v5_4000\comparison_v5_vs_v5_4000.json
status: ok
baseline_v5_not_overwritten: True

== Modele v5 ==
Fichier: c:\Users\Éleve\Pictures\Traine_aide_decission\outputs\readiness\bootstrap_model_v5\training_report_v5.json
status: ok
dataset_size: {'excluded':

In [76]:
from IPython.display import SVG, display, Markdown

heatmap_model_dir = V5_4000_MODEL_DIR if (V5_4000_MODEL_DIR / "heatmap_model_used_features.svg").exists() else V5_MODEL_DIR
for title, path in [
    ("V5_4000 - features utilisees par le modele", heatmap_model_dir / "heatmap_model_used_features.svg"),
    ("V5_4000 - features exclues car constantes", heatmap_model_dir / "heatmap_excluded_constant_features.svg"),
]:
    display(Markdown(f"### {title}"))
    if path.exists():
        display(SVG(filename=str(path)))
    else:
        print("Fichier absent:", path)

audit_only_path = heatmap_model_dir / "heatmap_audit_only_features.svg"
display(Markdown(f"### Heatmap audit-only non affichee par defaut
`{audit_only_path}`"))

for title, path in [
    ("Feature status summary", heatmap_model_dir / "feature_status_summary.md"),
    ("Logistic regression coefficients", heatmap_model_dir / "model_coefficients.md"),
    ("Ablation summary", heatmap_model_dir / "ablation_summary.md"),
]:
    display(Markdown(f"### {title}"))
    if path.exists():
        display(Markdown(path.read_text(encoding="utf-8")))
    else:
        print("Fichier absent:", path)


SyntaxError: unterminated f-string literal (detected at line 15) (1252862110.py, line 15)

In [ ]:
from IPython.display import SVG, display, Markdown

advanced_dir = V5_4000_MODEL_DIR
for title, path in [
    ("Correlation des inputs modele", advanced_dir / "input_feature_correlation_matrix.svg"),
    ("Learning curve", advanced_dir / "learning_curve.svg"),
]:
    display(Markdown(f"### {title}"))
    if path.exists():
        display(SVG(filename=str(path)))
    else:
        print("Fichier absent:", path)

for title, path in [
    ("High correlation pairs", advanced_dir / "high_correlation_pairs.md"),
    ("Multicollinearity report", advanced_dir / "multicollinearity_report.md"),
    ("Robust generalization report", advanced_dir / "robust_generalization_report.md"),
    ("Current vs BB normalized", V5_4000_MODEL_DIR.parent / "bootstrap_model_v5_4000_bb" / "comparison_current_vs_bb.md"),
]:
    display(Markdown(f"### {title}"))
    if path.exists():
        display(Markdown(path.read_text(encoding="utf-8")))
    else:
        print("Fichier absent:", path)


In [ ]:
active_candidates = V5_4000_CANDIDATES if V5_4000_CANDIDATES.exists() else V5_CANDIDATES
rows = load_csv(active_candidates)
kept = [row for row in rows if not is_true(row.get("excluded"))]
rejected = [row for row in rows if is_true(row.get("excluded"))]

def nested_count(rows, a, b):
    table = defaultdict(Counter)
    for row in rows:
        table[row.get(a) or "UNKNOWN"][row.get(b) or "UNKNOWN"] += 1
    return {key: dict(counts) for key, counts in sorted(table.items())}

print("Dataset:", active_candidates)
print("Total:", len(rows), "Kept:", len(kept), "Rejected:", len(rejected))
print("Labels:", dict(Counter(row.get("bootstrap_label") or row.get("labels.final_action") for row in kept)))
print("Sources:", dict(Counter(row.get("label_source") or row.get("metadata.label_source") for row in kept)))
print("Rejets:", dict(Counter(row.get("exclusion_reason") for row in rejected)))
print("Classes par source:", json.dumps(nested_count(kept, "metadata.label_source", "labels.final_action"), ensure_ascii=False, indent=2))
print("Classes par position:", json.dumps(nested_count(kept, "features.hero_position", "labels.final_action"), ensure_ascii=False, indent=2))


In [ ]:
contract_path = V5_4000_MODEL_DIR / "feature_contract.json"
if not contract_path.exists():
    contract_path = V5_MODEL_DIR / "feature_contract.json"

if contract_path.exists():
    contract = load_json(contract_path)
    print("Contrat:", contract_path)
    print("Schema:", contract.get("schema"))
    print("Features modele:", len(contract.get("feature_order", [])))
    for feature in contract.get("feature_order", []):
        print("-", feature)
    print("Colonnes ignorees:", len(contract.get("ignored_columns", [])))
    print("Leakage exclu:", contract.get("leakage_columns_excluded"))
else:
    print("Contrat absent:", contract_path)


## Lecture recommandee

Avant d'ajouter `CALL`, verifier manuellement quelques lignes du dataset actif. Si les spots semblent artificiels ou trop homogenes, il faut diversifier les scenarios avant de faire grossir le dataset.
